# Elevation Analysis Pipeline - Testing Notebook
Complete end-to-end test with parametrized code

**Goal**: Replicate original Oct 23 pipeline logic with new fully-parametrized codebase

## Cell 1: Setup Path & Imports

In [1]:
import os, sys, json, math, inspect
import numpy as np
import pandas as pd
import importlib
import pathlib


# ===== PATH SETUP =====
# Notebook is in: <repo>/uchicago-elevation/scripts/
# Project root is: <repo>/uchicago-elevation/
# Source code is: <repo>/uchicago-elevation/src/

NB_DIR = os.getcwd()
PROJ_ROOT = os.path.dirname(NB_DIR)  # Go up one level to uchicago-elevation/
SRC = os.path.join(PROJ_ROOT, "src")

# Ensure src is at front of sys.path
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

# ===== DIAGNOSTIC OUTPUT =====
print("="*80)
print("PATH DIAGNOSTICS")
print("="*80)
print(f"Notebook dir:     {NB_DIR}")
print(f"Project root:     {PROJ_ROOT}")
print(f"SRC dir:          {SRC}")
print(f"SRC exists?       {os.path.exists(SRC)}")
print(f"\nKey files:")
print(f"  dem.py:         {os.path.exists(os.path.join(SRC, 'elevation', 'dem.py'))}")
print(f"  osm.py:         {os.path.exists(os.path.join(SRC, 'elevation', 'osm.py'))}")
print(f"  terrain.py:     {os.path.exists(os.path.join(SRC, 'elevation', 'terrain.py'))}")
print(f"  viz.py:         {os.path.exists(os.path.join(SRC, 'elevation', 'viz.py'))}")
print(f"  pipeline.py:    {os.path.exists(os.path.join(SRC, 'elevation', 'pipeline.py'))}")
print(f"  __init__.py:    {os.path.exists(os.path.join(SRC, 'elevation', '__init__.py'))}")

# Set outputs directory
outputs_dir = os.path.join(PROJ_ROOT, "outputs")
os.makedirs(outputs_dir, exist_ok=True)
print(f"\nOutputs dir:      {outputs_dir}")
print("="*80)

PATH DIAGNOSTICS
Notebook dir:     /Users/yaoyiyang/Desktop/Uchicago-elevation/scripts
Project root:     /Users/yaoyiyang/Desktop/Uchicago-elevation
SRC dir:          /Users/yaoyiyang/Desktop/Uchicago-elevation/src
SRC exists?       True

Key files:
  dem.py:         True
  osm.py:         True
  terrain.py:     True
  viz.py:         True
  pipeline.py:    True
  __init__.py:    True

Outputs dir:      /Users/yaoyiyang/Desktop/Uchicago-elevation/outputs


## Cell 2: Import All Modules

In [2]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from elevation.run_final_pipeline import PipelineConfig, run_pipeline


In [3]:
import os, sys
# 让 Python 能找到 src/
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

print("\n" + "="*80)
print("IMPORTING MODULES (FINAL)")
print("="*80)

# —— 都用包名导入；不要任何 try/except 兜底 ——
from elevation.osm import fetch_building_osm
print("✓ elevation.osm")

from elevation.dem import (
    discover_usgs_lidar_dataset, build_multiple_dems, load_dem, build_all_circle_masks
)
print("✓ elevation.dem")

from elevation.terrain import (
    estimate_house_ground_adaptive, derive_slope_aspect_curvature, classify_terrain_adaptive
)
print("✓ elevation.terrain")

from elevation.analytics import (
    ring_metrics_for_radii, build_multiscale_summary_df, area_level_summary
)
print("✓ elevation.analytics")

from elevation.viz import (
    figure1_elevation, figure2_slope, figure3_aspect, figure4_terrain_and_hist,
    add_3d_figures_to_pipeline, DEFAULT_CONFIG
)
print("✓ elevation.viz")

from elevation.run_final_pipeline import PipelineConfig, run_pipeline
print("✓ elevation.run_final_pipeline")

print("\n✅ ALL IMPORTS SUCCESSFUL")
print("="*80)



IMPORTING MODULES (FINAL)
✓ elevation.osm
✓ elevation.dem
✓ elevation.terrain
✓ elevation.analytics
✓ elevation.viz
✓ elevation.run_final_pipeline

✅ ALL IMPORTS SUCCESSFUL


## Cell 3: Test Location Input (Example 1: Naples, FL)

In [4]:
print("\n" + "="*80)
print("TEST 1: NAPLES, FLORIDA")
print("="*80)

# ===== USER INPUT =====
test_lat = 26.409538
test_lon = -81.784861
test_address = "Naples, FL"

# ===== CREATE CONFIG =====
config = PipelineConfig()

# Set location
config.lat = test_lat
config.lon = test_lon
config.address = test_address

# ===== CUSTOMIZE PARAMETERS (ALL PARAMETRIZED!) =====
config.aoi_radius_m = 100.0               # User-selectable AOI
config.osm_buffer_m = 120.0               # OSM search radius
config.dem_resolution_m = 1.0
config.dem_for_user_aoi = True
config.reference_radius_m = 500.0

# Ring metrics radii (parametrized!)
config.ring_metrics_radii = [50.0, 200.0, 500.0]

# Terrain thresholds (parametrized!)
config.slope_low_threshold = 2.0
config.slope_mod_threshold = 5.0

# House estimation (all parametrized!)
config.min_size_edge = 10
config.min_size_ring_3_8 = 20
config.min_size_ring_8_15 = 30
config.min_size_ring_15_30 = 50
config.q_high_tail_threshold = 98.0
config.hard_q_high_tail = 99.5

# Output settings
config.output_dir = outputs_dir
config.output_format = "csv"             # parquet + csv
config.generate_narrative = True
config.save_3d = True
config.verbose = True

print(f"\nConfig Summary:")
print(f"  Location:              {config.lat}, {config.lon}")
print(f"  AOI radius:            {config.aoi_radius_m}m (user-selectable!)")
print(f"  OSM buffer:            {config.osm_buffer_m}m")
print(f"  Ring metrics radii:    {config.ring_metrics_radii}")
print(f"  Slope thresholds:      {config.slope_low_threshold}°, {config.slope_mod_threshold}°")
print(f"  Output dir:            {config.output_dir}")
print("="*80)


TEST 1: NAPLES, FLORIDA

Config Summary:
  Location:              26.409538, -81.784861
  AOI radius:            100.0m (user-selectable!)
  OSM buffer:            120.0m
  Ring metrics radii:    [50.0, 200.0, 500.0]
  Slope thresholds:      2.0°, 5.0°
  Output dir:            /Users/yaoyiyang/Desktop/Uchicago-elevation/outputs


## Cell 4: Run Pipeline (Main Execution)

In [5]:
print("\n" + "="*80)
print("EXECUTING PIPELINE")
print("="*80)

try:
    results = run_pipeline(config)
    print("\n✅ PIPELINE EXECUTION SUCCESSFUL")
except Exception as e:
    print(f"\n❌ PIPELINE FAILED: {e}")
    import traceback
    traceback.print_exc()
    raise


EXECUTING PIPELINE
TERRAIN ANALYSIS PIPELINE - AOI RADIUS: 100.0m
Target location: 26.409538, -81.784861

STEP 1: LIDAR DATASET DISCOVERY
Dataset: FL_LeeCounty_2007

STEP 2: DEM GENERATION (DTM + DSM)
DEM specifications:
  - 500m reference: 1.0m resolution
  - 100.0m user: 2.0m resolution
  🚀 复用缓存的 500.0m DEM (DTM+DSM)
  🚀 复用缓存的 100.0m DEM (DTM+DSM)
Generated 2 DEM pairs (DTM + DSM)

STEP 3: LOADING DEM & BUILDING MASKS
User DEM (100.0m): (100, 100)
Reference DEM (500m): (1000, 1000)
DEM shape: (1000, 1000)
Primary mask (100.0m): 31424 pixels

STEP 4: FETCHING OSM BUILDING

STEP 5: COMPUTING TERRAIN DERIVATIVES
✓ Computed slope, aspect, curvature

STEP 6: ESTIMATING HOUSE ELEVATION
House elevation: 4.80 m (method: median_IQRtrim_k1.8)

STEP 7: TERRAIN CLASSIFICATION
✓ Terrain classified

STEP 8: GENERATING VISUALIZATIONS
✓ figure1_elevation.png
✓ figure2_slope.png
✓ figure3_aspect.png
✓ figure4_terrain_and_hist.png
  开始叠加多边形到4个子图...
  ✓ Saved figure5_6_3d_combo.html
✔ 3D visualization

## Cell 5: Display Results Summary

In [6]:
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

print(f"\n📍 Location: {config.lat:.6f}, {config.lon:.6f}")
print(f"🏠 House Elevation: {results['house_ground_m']:.2f} m")
print(f"📊 Percentile Rank: {results['percentile_rank']:.1f}%" if results['percentile_rank'] is not None else "📊 Percentile Rank: N/A")
print(f"🗂️  Dataset: {results['dataset']}")
print(f"⭕ AOI Radius: {results['aoi_radius_m']}m")
print(f"📁 Output Dir: {results['outputs_dir']}")

print(f"\n📋 Ring Metrics Summary:")
display(results['summary_multiscale'])

print(f"\n📈 Area Statistics:")
display(results['summary_area_level'])


RESULTS SUMMARY

📍 Location: 26.409538, -81.784861
🏠 House Elevation: 4.80 m
📊 Percentile Rank: 43.0%
🗂️  Dataset: FL_LeeCounty_2007
⭕ AOI Radius: 100.0m
📁 Output Dir: /Users/yaoyiyang/Desktop/Uchicago-elevation/outputs/26p409538_-81p784861_100m

📋 Ring Metrics Summary:


,Radius (m),Pixels,ΔElev_median (m),% Higher,% Lower,Slope_mean (°),Slope_median (°),Slope_P25 (°),Slope_P75 (°),% Flat <2°,% Gentle 2–5°,% Steep ≥5°,Convergence (%),Dominant Aspect (°),Dominant Aspect (cardinal)
0,50.0,7856,0.011,38.3,40.1,3.99,2.84,1.33,5.34,26.4,26.0,19.9,30.2,352.0,N
1,200.0,125667,-0.063,34.4,27.9,5.98,3.18,1.44,7.52,19.7,16.0,20.6,32.3,357.1,N
2,500.0,785399,-0.246,51.8,24.6,6.00,3.35,1.52,7.67,24.0,20.4,27.0,31.5,354.3,N



📈 Area Statistics:


,Metric,Value,Interpretation
0,House Elevation,4.80 m,Reference elevation
1,Lowest Elevation in 500m,2.86 m,1.94 m above lowest
2,Highest Elevation in 500m,16.13 m,11.33 m below highest
3,Median Elevation in 500m,5.04 m,Below median
4,Elevation Percentile Rank in 500m,32.15%,Below regional median


## Cell 7: Verify Outputs Match Oct 23 Logic

## Cell 8: Display Ring Metrics Table (Detailed)

In [7]:
print("\n" + "="*80)
print("DETAILED RING METRICS")
print("="*80)

df = results['summary_multiscale']
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

display(df)

print(f"\n📊 Statistics:")
for col in df.select_dtypes(include='number').columns:
    print(f"  {col}:")
    print(f"    Min:    {df[col].min():.2f}")
    print(f"    Max:    {df[col].max():.2f}")
    print(f"    Mean:   {df[col].mean():.2f}")


DETAILED RING METRICS


,Radius (m),Pixels,ΔElev_median (m),% Higher,% Lower,Slope_mean (°),Slope_median (°),Slope_P25 (°),Slope_P75 (°),% Flat <2°,% Gentle 2–5°,% Steep ≥5°,Convergence (%),Dominant Aspect (°),Dominant Aspect (cardinal)
0,50.0,7856,0.011,38.3,40.1,3.99,2.84,1.33,5.34,26.4,26.0,19.9,30.2,352.0,N
1,200.0,125667,-0.063,34.4,27.9,5.98,3.18,1.44,7.52,19.7,16.0,20.6,32.3,357.1,N
2,500.0,785399,-0.246,51.8,24.6,6.00,3.35,1.52,7.67,24.0,20.4,27.0,31.5,354.3,N



📊 Statistics:
  Radius (m):
    Min:    50.00
    Max:    500.00
    Mean:   250.00
  Pixels:
    Min:    7856.00
    Max:    785399.00
    Mean:   306307.33
  ΔElev_median (m):
    Min:    -0.25
    Max:    0.01
    Mean:   -0.10
  % Higher:
    Min:    34.40
    Max:    51.80
    Mean:   41.50
  % Lower:
    Min:    24.60
    Max:    40.10
    Mean:   30.87
  Slope_mean (°):
    Min:    3.99
    Max:    6.00
    Mean:   5.32
  Slope_median (°):
    Min:    2.84
    Max:    3.35
    Mean:   3.12
  Slope_P25 (°):
    Min:    1.33
    Max:    1.52
    Mean:   1.43
  Slope_P75 (°):
    Min:    5.34
    Max:    7.67
    Mean:   6.84
  % Flat <2°:
    Min:    19.70
    Max:    26.40
    Mean:   23.37
  % Gentle 2–5°:
    Min:    16.00
    Max:    26.00
    Mean:   20.80
  % Steep ≥5°:
    Min:    19.90
    Max:    27.00
    Mean:   22.50
  Convergence (%):
    Min:    30.20
    Max:    32.30
    Mean:   31.33
  Dominant Aspect (°):
    Min:    352.00
    Max:    357.10
    Mean:   354.47
